# Production ledger grain at 24ff1ec90
Read-only check. Run inside the repository with this commit available. Counts cover committed records. A wind record is an attempt, and several can occupy a single scheduled hour.

In [1]:
from collections import Counter
from pathlib import Path
import csv
import json
import subprocess

root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
snapshot = '24ff1ec90'
def source(path):
    return subprocess.check_output(['git', 'show', f'{snapshot}:{path}'], cwd=root, text=True)

for name, columns in [
    ('predictions_log', (0, 1)), ('outlook_log', (0, 1)),
    ('forecast_accuracy', (0,)), ('day_risk_log', (0, 1)),
    ('observed_peaks_cache', (0,)), ('nowcast_heartbeats', (0, 1))
]:
    rows = list(csv.reader(source(f'data/{name}.csv').splitlines(), strict=True))[1:]
    keys = Counter(tuple(row[i] for i in columns) for row in rows)
    print(name, len(rows), 'rows', len(rows) - len(keys), 'excess keys')

for name in ('replay_inputs', 'wind_shadow'):
    paths = subprocess.check_output(['git', 'ls-tree', '-r', '--name-only', snapshot,
                                     f'data/{name}'], cwd=root, text=True).splitlines()
    records = [json.loads(line) for path in paths if path.endswith('.jsonl')
               for line in source(path).splitlines()]
    print(name, len(records), dict(Counter(r.get('status', 'record') for r in records)))


predictions_log 24482 rows 0 excess keys
outlook_log 21057 rows 0 excess keys
forecast_accuracy 184 rows 0 excess keys
day_risk_log 1743 rows 0 excess keys
observed_peaks_cache 370 rows 0 excess keys
nowcast_heartbeats 1086 rows 0 excess keys
replay_inputs 1533 {'record': 1533}
wind_shadow 1522 {'candidate': 497, 'fallback': 1025}
